# GAT-v4 | Phase 5 — Graph Construction

**Covers plan sections 7.1–7.4**

### What this notebook does
1. **PPI subgraph** — filter STRING v12 to DEG nodes (combined_score ≥ 600), make undirected
2. **Pearson edge weights** — per fold, computed on training tumor VST only (no leakage)
3. **Reactome pathway mapping** — Ensembl2Reactome → HGNC symbols → DEG filter → Jaccard dedup

### Key leakage prevention
Pearson correlations are computed **only on training tumor samples** per fold.
Test and validation samples never touch the edge weight computation.

### Expected outputs
- `data/processed/ppi_edges.csv` — base undirected edge list (gene1, gene2, string_score)
- `data/processed/reactome_filtered.csv` — pathway → gene mapping after Jaccard dedup
- `data/splits/fold_<seed>_<fold>/edge_index.npy` — per-fold COO edge indices
- `data/splits/fold_<seed>_<fold>/edge_weight.npy` — per-fold Pearson |r| weights

---
| Cell | What |
|------|------|
| 1 | Imports & paths |
| 2 | Load DEG list, VST matrix, CV splits |
| 3 | Locate STRING files |
| 4 | Load STRING aliases → gene symbol map |
| 5 | Load STRING links → filter + map to symbols |
| 6 | Build PPI subgraph (DEG nodes only) |
| 7 | Pearson edge weights — vectorised, per fold |
| 8 | Save per-fold edge tensors |
| 9 | Locate & load Reactome files |
| 10 | Map Reactome Ensembl → HGNC → DEG filter |
| 11 | Jaccard redundancy filtering |
| 12 | Save outputs + audit YAML |

---
## Cell 1 — Imports & Paths

In [9]:
import sys, json, yaml, hashlib, time, gzip
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT   = PROJECT_ROOT / "KIRC"
DATA_PROC    = KIRC_ROOT / "data" / "processed"
DATA_SPLITS  = KIRC_ROOT / "data" / "splits"
AUDIT_DIR    = KIRC_ROOT / "audit"

STRING_DIR   = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\STRING")
REACTOME_DIR = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\REACTOME")
# ─────────────────────────────────────────────────────────────────────────────

if str(KIRC_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRC_ROOT))
from src.utils.logger import get_logger
from src.config import STRING_SCORE_THRESHOLD, PEARSON_EDGE_THRESHOLD, JACCARD_REDUNDANCY
from src.config import PATHWAY_MIN_GENES, PATHWAY_MAX_GENES

log = get_logger('phase5_graph')

print(f"STRING score threshold  : >= {STRING_SCORE_THRESHOLD}")
print(f"Pearson edge threshold  : |r| >= {PEARSON_EDGE_THRESHOLD}")
print(f"Jaccard dedup threshold : < {JACCARD_REDUNDANCY}")
print(f"Pathway size filter     : [{PATHWAY_MIN_GENES}, {PATHWAY_MAX_GENES}] DEG members")
print()
for label, p in [("STRING dir", STRING_DIR), ("REACTOME dir", REACTOME_DIR)]:
    print(f"  {'OK  ' if p.exists() else 'MISS'} {label}: {p}")

C:\Users\jiyad\Desktop\PROJECT\KIRC\script\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-06 23:05:53 | INFO     | phase5_graph | Logger initialised. Log file → C:\Users\jiyad\OneDrive\Desktop\PROJECT\gat-v4\results\logs\phase5_graph_20260706_230553.log


STRING score threshold  : >= 600
Pearson edge threshold  : |r| >= 0.3
Jaccard dedup threshold : < 0.7
Pathway size filter     : [5, 500] DEG members

  OK   STRING dir: C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\STRING
  OK   REACTOME dir: C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\REACTOME


---
## Cell 2 — Load DEG List, VST Matrix, CV Splits

In [10]:
# ── DEG symbols (unique, same as used in Phase 3+4) ──────────────────────────
deg_res = pd.read_csv(DATA_PROC / 'deg_results.csv', index_col=0)
deg_mask = (
    deg_res['padj'].notna() &
    (deg_res['padj'] < 0.05) &
    (deg_res['log2FoldChange'].abs() > 1.0)
)
deg_symbols_raw = deg_res[deg_mask].index.tolist()
# Deduplicate (same step as Phase 3)
seen = set()
deg_symbols = [g for g in deg_symbols_raw if not (g in seen or seen.add(g))]
deg_set     = set(deg_symbols)
print(f"DEG symbols (unique) : {len(deg_symbols):,}")

# ── VST matrix ────────────────────────────────────────────────────────────────
print("Loading VST matrix...")
t0 = time.time()
vst = pd.read_csv(DATA_PROC / 'vst_matrix.csv', index_col=0)
print(f"  VST shape : {vst.shape}  ({time.time()-t0:.0f}s)")
print(f"  Index sample : {list(vst.index[:4])}")
print(f"  Col sample   : {list(vst.columns[:3])}")

# ── Sample alignment (to identify tumor sample IDs) ──────────────────────────
align       = pd.read_csv(DATA_PROC / 'sample_alignment.csv')
tumor_sids  = set(align[(align['tissue_type']=='tumor')  & align['include']]['sample_id'])
normal_sids = set(align[(align['tissue_type']=='normal') & align['include']]['sample_id'])
print(f"  Tumor samples  : {len(tumor_sids)}")
print(f"  Normal samples : {len(normal_sids)}")

# ── CV splits ─────────────────────────────────────────────────────────────────
with open(DATA_SPLITS / 'cv_folds_repeated_5x3.json') as f:
    cv_splits = json.load(f)
print(f"\nCV folds loaded : {len(cv_splits)}")

# Patient → sample ID map (tumor only, for finding VST columns per fold)
align_tumor  = align[(align['tissue_type']=='tumor') & align['include']]
pat_to_sids  = align_tumor.groupby('patient_id')['sample_id'].apply(list).to_dict()

log.info(f"DEGs: {len(deg_symbols)}, VST: {vst.shape}, CV folds: {len(cv_splits)}")
print("\nCell 2 done.")

DEG symbols (unique) : 9,997
Loading VST matrix...


2026-07-06 23:05:57 | INFO     | phase5_graph | DEGs: 9997, VST: (9997, 569), CV folds: 15


  VST shape : (9997, 569)  (3s)
  Index sample : ['TSPAN6', 'FGR', 'SEMA3F', 'CYP51A1']
  Col sample   : ['TCGA-B8-4146-01B', 'TCGA-BP-4977-01A', 'TCGA-BP-4981-01A']
  Tumor samples  : 496
  Normal samples : 69

CV folds loaded : 15

Cell 2 done.


---
## Cell 3 — Locate STRING Files

STRING v12 files expected (may be gzipped or not):
- `9606.protein.links.v12.0.txt[.gz]` — PPI links with scores
- `9606.protein.aliases.v12.0.txt[.gz]` — protein ID → gene symbol mapping

In [11]:
def find_file(directory, patterns):
    """Find first file matching any of the glob patterns."""
    for pat in patterns:
        matches = list(directory.glob(pat))
        if matches:
            return matches[0]
    return None

print(f"Scanning STRING dir: {STRING_DIR}")
print("Files found:")
for f in sorted(STRING_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1024**2:.0f} MB)")

STRING_LINKS   = find_file(STRING_DIR, [
    "9606.protein.links.v12.0.txt",
    "9606.protein.links.v12.0.txt.gz",
    "*protein.links*",
])
STRING_ALIASES = find_file(STRING_DIR, [
    "9606.protein.aliases.v12.0.txt",
    "9606.protein.aliases.v12.0.txt.gz",
    "*protein.aliases*",
])

print()
print(f"Links file   : {STRING_LINKS}")
print(f"Aliases file : {STRING_ALIASES}")

if not STRING_LINKS or not STRING_ALIASES:
    print("\nMISSING files. Check STRING_DIR path or file names above.")
else:
    print("\nBoth STRING files found.")

Scanning STRING dir: C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\STRING
Files found:
  9606.protein.aliases.v12.0.txt  (190 MB)
  9606.protein.info.v12.0.txt  (6 MB)
  9606.protein.links.v12.0.txt  (602 MB)

Links file   : C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\STRING\9606.protein.links.v12.0.txt
Aliases file : C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\STRING\9606.protein.aliases.v12.0.txt

Both STRING files found.


---
## Cell 4 — Load STRING Aliases → Gene Symbol Map

STRING protein IDs (e.g. `9606.ENSP00000000233`) → HGNC gene symbols.

The alias file has sources including `Ensembl_HGNC`, `BLAST_UniProt_GN_Name`, etc.
We scan all sources and keep the first gene-symbol-like alias per protein.

In [12]:
# Cell 4 — STRING Aliases → Gene Symbol Map (FIXED with correct source names)

is_gz = str(STRING_ALIASES).endswith('.gz')
aliases = pd.read_csv(
    STRING_ALIASES,
    sep='\t',
    compression='gzip' if is_gz else None,
    dtype=str, header=0
)
aliases.columns = [c.lstrip('#').strip() for c in aliases.columns]
aliases = aliases.rename(columns={
    'string_protein_id': 'protein_id',
    'alias': 'alias',
    'source': 'source'
})

# ── Use only the two confirmed HGNC symbol sources ───────────────────────────
# Priority: Ensembl_HGNC_symbol first, then BioMart_HUGO as fallback
SYMBOL_SOURCES = ['Ensembl_HGNC_symbol', 'BioMart_HUGO', 'UniProt_GN_Name']

sym_aliases = aliases[aliases['source'].isin(SYMBOL_SOURCES)].copy()
print(f"Rows from symbol sources : {len(sym_aliases):,}")
print(sym_aliases['source'].value_counts().to_string())

# Sort so Ensembl_HGNC_symbol rows come first (highest priority)
source_priority = {s: i for i, s in enumerate(SYMBOL_SOURCES)}
sym_aliases['_priority'] = sym_aliases['source'].map(source_priority)
sym_aliases = sym_aliases.sort_values('_priority')

# One symbol per protein_id — keep highest priority source
protein_to_symbol = (
    sym_aliases
    .drop_duplicates('protein_id', keep='first')
    .set_index('protein_id')['alias']
    .to_dict()
)

print(f"\nProtein → symbol map : {len(protein_to_symbol):,} entries")

# Spot-check: how many of our DEGs are reachable?
all_symbols_in_map = set(protein_to_symbol.values())
degs_reachable = deg_set & all_symbols_in_map
print(f"DEG symbols reachable via map : {len(degs_reachable):,} / {len(deg_set):,}")
print(f"Examples: {list(protein_to_symbol.items())[:6]}")

log.info(f"STRING alias map: {len(protein_to_symbol)} proteins, {len(degs_reachable)} DEGs reachable")
print("\nCell 4 done.")

2026-07-06 23:06:04 | INFO     | phase5_graph | STRING alias map: 19409 proteins, 5448 DEGs reachable


Rows from symbol sources : 57,773
source
UniProt_GN_Name        19290
BioMart_HUGO           19280
Ensembl_HGNC_symbol    19203

Protein → symbol map : 19,409 entries
DEG symbols reachable via map : 5,448 / 9,997
Examples: [('9606.ENSP00000501201', 'TCP11'), ('9606.ENSP00000002829', 'SEMA3F'), ('9606.ENSP00000002596', 'HS3ST1'), ('9606.ENSP00000002165', 'FUCA2'), ('9606.ENSP00000002125', 'NDUFAF7'), ('9606.ENSP00000004531', 'SLC7A2')]

Cell 4 done.


---
## Cell 5 — Load STRING Links → Filter + Map to Symbols

Filters to `combined_score >= 600` then maps protein IDs to gene symbols.

**Runtime: ~1–3 min** (links file is ~500 MB).

In [13]:
is_gz_links = str(STRING_LINKS).endswith('.gz')
print(f"Loading STRING links ({'gzipped' if is_gz_links else 'plain'})...")
t0 = time.time()

links = pd.read_csv(
    STRING_LINKS,
    sep=' ',
    compression='gzip' if is_gz_links else None,
    dtype={'protein1': str, 'protein2': str, 'combined_score': int}
)
print(f"  All links      : {len(links):,}  ({time.time()-t0:.0f}s)")
print(f"  Columns        : {list(links.columns)}")

# Filter by combined_score
links_filt = links[links['combined_score'] >= STRING_SCORE_THRESHOLD].copy()
print(f"  After score>={STRING_SCORE_THRESHOLD} : {len(links_filt):,}")
del links  # free ~2 GB

# Map protein IDs to gene symbols
links_filt['gene1'] = links_filt['protein1'].map(protein_to_symbol)
links_filt['gene2'] = links_filt['protein2'].map(protein_to_symbol)

n_before_drop = len(links_filt)
links_filt = links_filt.dropna(subset=['gene1', 'gene2'])
print(f"  After symbol map (drop unmapped) : {len(links_filt):,}")
print(f"  Dropped (no symbol)              : {n_before_drop - len(links_filt):,}")

# Show mapping rate
p1_mapped = links_filt['protein1'].nunique()
p2_mapped = links_filt['protein2'].nunique()
print(f"  Unique proteins with symbols     : {max(p1_mapped, p2_mapped):,}")

log.info(f"STRING links loaded: {len(links_filt)} after score filter + symbol mapping")
print("\nCell 5 done.")

Loading STRING links (plain)...
  All links      : 13,715,404  (12s)
  Columns        : ['protein1', 'protein2', 'combined_score']
  After score>=600 : 720,388
  After symbol map (drop unmapped) : 715,298
  Dropped (no symbol)              : 5,090


2026-07-06 23:06:17 | INFO     | phase5_graph | STRING links loaded: 715298 after score filter + symbol mapping


  Unique proteins with symbols     : 17,893

Cell 5 done.


---
## Cell 6 — Build PPI Subgraph (DEG Nodes Only)

- Restrict to edges where **both** endpoints are DEGs
- Make undirected (sort gene pairs, deduplicate)
- Build gene → integer index mapping

Expected: **3,000–5,000 nodes**, **15,000–35,000 edges** (plan expected 4,000–4,500 nodes;
our higher DEG count may push this higher).

In [14]:
# Cell 6 — Build PPI Subgraph (FIXED — no duplicate column names)

# Restrict to DEG nodes
ppi = links_filt[
    links_filt['gene1'].isin(deg_set) &
    links_filt['gene2'].isin(deg_set)
][['gene1', 'gene2', 'combined_score']].copy()

print(f"Edges (both endpoints DEG) : {len(ppi):,}")

# Remove self-loops
ppi = ppi[ppi['gene1'] != ppi['gene2']].copy()
print(f"After removing self-loops  : {len(ppi):,}")

# Make undirected: create sorted pair WITHOUT renaming over existing columns
sorted_pairs = ppi.apply(
    lambda r: (r['gene1'], r['gene2']) if r['gene1'] < r['gene2']
              else (r['gene2'], r['gene1']),
    axis=1
)
ppi = ppi.copy()
ppi['g1'] = [p[0] for p in sorted_pairs]
ppi['g2'] = [p[1] for p in sorted_pairs]
ppi['edge_id'] = ppi['g1'] + '|' + ppi['g2']

# Keep highest score per undirected edge
ppi = (
    ppi.sort_values('combined_score', ascending=False)
       .drop_duplicates('edge_id', keep='first')
       [['g1', 'g2', 'combined_score']]
       .rename(columns={'g1': 'gene1', 'g2': 'gene2'})
       .reset_index(drop=True)
)

print(f"Undirected unique edges    : {len(ppi):,}")

# Node set and index mapping — use .values.tolist() to avoid Series confusion
ppi_genes   = sorted(set(ppi['gene1'].values.tolist() + ppi['gene2'].values.tolist()))
gene_to_idx = {g: i for i, g in enumerate(ppi_genes)}

print(f"Unique DEG nodes in PPI    : {len(ppi_genes):,}  (expected 3,000–5,000)")
print(f"Edges                      : {len(ppi):,}  (expected 15,000–35,000)")

if len(ppi) < 1000:
    print()
    print("WARNING: Very few edges. The alias mapping in Cell 4 likely failed.")
    print("  Re-run Cell 4 with the fixed diagnostic version and check source names.")

Edges (both endpoints DEG) : 75,776
After removing self-loops  : 75,768
Undirected unique edges    : 37,850
Unique DEG nodes in PPI    : 4,414  (expected 3,000–5,000)
Edges                      : 37,850  (expected 15,000–35,000)


---
## Cell 7 — Pearson Edge Weights (Vectorised, Per Fold)

For each of the 15 folds:
1. Get training tumor sample IDs from CV splits
2. Subset VST to those samples (training tumor only — no leakage)
3. Compute |Pearson r| for every PPI edge using vectorised numpy
4. Prune edges with |r| < 0.3

**Vectorised approach:** extract the two gene expression vectors for all edges
simultaneously using index arrays, then compute correlation in batch.

**Runtime: ~5–15 min total** (~20–60 sec per fold).

In [15]:
# Pre-build edge arrays in gene index space (these don't change across folds)
ppi_gene1_idx = np.array([gene_to_idx[g] for g in ppi['gene1']])
ppi_gene2_idx = np.array([gene_to_idx[g] for g in ppi['gene2']])
n_edges = len(ppi)
print(f"Edge index arrays built: {n_edges:,} edges")

# VST submatrix for PPI genes only (genes x samples)
# Reindex VST to PPI gene order for fast numpy indexing
ppi_genes_in_vst = [g for g in ppi_genes if g in vst.index]
ppi_genes_missing_vst = [g for g in ppi_genes if g not in vst.index]
print(f"PPI genes in VST       : {len(ppi_genes_in_vst):,} / {len(ppi_genes):,}")
if ppi_genes_missing_vst:
    print(f"  Missing from VST (will use mean imputation): {len(ppi_genes_missing_vst)}")

# Build gene → VST row index map
vst_ppi = vst.reindex(ppi_genes)   # shape: (n_ppi_genes x n_samples), NaN for missing
print(f"VST PPI submatrix: {vst_ppi.shape}")

fold_results = {}  # fold_key -> {'edge_index': ..., 'edge_weight': ...}

t_total = time.time()
for fold_key, fold in tqdm(cv_splits.items(), desc='Computing Pearson weights'):
    t0 = time.time()

    # Get training tumor sample IDs for this fold
    train_pats = fold['train_patients']
    train_sids = []
    for pat in train_pats:
        sids = pat_to_sids.get(pat, [])
        train_sids.extend([s for s in sids if s in vst.columns])

    if len(train_sids) < 10:
        log.warning(f"{fold_key}: only {len(train_sids)} training tumor samples in VST")
        continue

    # VST values for training tumor samples — shape: (n_ppi_genes, n_train_samples)
    vst_train = vst_ppi[train_sids].values.astype(np.float32)

    # Mean-impute missing genes (NaN rows) with row mean of present samples
    row_means = np.nanmean(vst_train, axis=1, keepdims=True)
    nan_mask  = np.isnan(vst_train)
    vst_train[nan_mask] = np.broadcast_to(row_means, vst_train.shape)[nan_mask]

    # Vectorised Pearson |r| for all edges
    # Extract expression vectors for gene1 and gene2 endpoints
    X = vst_train[ppi_gene1_idx]   # (n_edges, n_train_samples)
    Y = vst_train[ppi_gene2_idx]   # (n_edges, n_train_samples)

    # Centre each row (per-gene mean subtraction)
    X_c = X - X.mean(axis=1, keepdims=True)
    Y_c = Y - Y.mean(axis=1, keepdims=True)

    # Pearson r = dot(X_c, Y_c) / (||X_c|| * ||Y_c||)
    num   = (X_c * Y_c).sum(axis=1)
    denom = (np.linalg.norm(X_c, axis=1) * np.linalg.norm(Y_c, axis=1))

    # Safe division — if denom=0 (constant gene), r=0
    r = np.where(denom > 1e-8, num / denom, 0.0)
    abs_r = np.abs(r)

    # Prune edges with |r| < threshold
    keep_mask  = abs_r >= PEARSON_EDGE_THRESHOLD
    kept_g1    = ppi_gene1_idx[keep_mask]
    kept_g2    = ppi_gene2_idx[keep_mask]
    kept_w     = abs_r[keep_mask]

    # COO format: shape (2, n_kept_edges)
    edge_index  = np.stack([kept_g1, kept_g2], axis=0).astype(np.int32)
    edge_weight = kept_w.astype(np.float32)

    fold_results[fold_key] = {
        'edge_index'    : edge_index,
        'edge_weight'   : edge_weight,
        'n_train_sids'  : len(train_sids),
        'n_edges_before': n_edges,
        'n_edges_after' : int(keep_mask.sum()),
        'mean_abs_r'    : float(abs_r.mean()),
    }

print(f"\nTotal time : {time.time()-t_total:.0f}s")
print(f"\nFold-level edge stats:")
print(f"{'Fold':<25} {'Train_N':>8} {'Edges_kept':>12} {'Keep%':>7} {'Mean|r|':>9}")
for fk, res in fold_results.items():
    pct = res['n_edges_after'] / res['n_edges_before'] * 100
    print(f"{fk:<25} {res['n_train_sids']:>8} {res['n_edges_after']:>12,} {pct:>6.1f}% {res['mean_abs_r']:>9.3f}")

log.info(f"Pearson weights computed for {len(fold_results)} folds")
print("\nCell 7 done.")

Edge index arrays built: 37,850 edges
PPI genes in VST       : 4,414 / 4,414
VST PPI submatrix: (4414, 569)



Computing Pearson weights: 100%|██████████| 15/15 [00:04<00:00,  3.57it/s]
2026-07-06 23:06:24 | INFO     | phase5_graph | Pearson weights computed for 15 folds



Total time : 4s

Fold-level edge stats:
Fold                       Train_N   Edges_kept   Keep%   Mean|r|
seed42_fold0                   361       14,652   38.7%     0.298
seed42_fold1                   360       14,464   38.2%     0.296
seed42_fold2                   360       14,535   38.4%     0.296
seed42_fold3                   362       14,528   38.4%     0.297
seed42_fold4                   357       14,085   37.2%     0.289
seed123_fold0                  361       14,232   37.6%     0.292
seed123_fold1                  360       14,065   37.2%     0.290
seed123_fold2                  357       14,574   38.5%     0.296
seed123_fold3                  360       14,367   38.0%     0.294
seed123_fold4                  360       14,516   38.4%     0.298
seed456_fold0                  361       14,587   38.5%     0.297
seed456_fold1                  361       13,972   36.9%     0.288
seed456_fold2                  360       14,404   38.1%     0.295
seed456_fold3                  356 

---
## Cell 8 — Save Per-Fold Edge Tensors

Each fold gets its own subdirectory under `data/splits/`.
Saved as numpy `.npy` files (fast load with `np.load`).

In [16]:
# Also save the gene index mapping — needed in all downstream phases
gene_index_df = pd.DataFrame(
    {'gene_symbol': ppi_genes, 'node_idx': range(len(ppi_genes))}
)
gene_index_path = DATA_PROC / 'ppi_gene_index.csv'
gene_index_df.to_csv(gene_index_path, index=False)
print(f"Gene index map -> {gene_index_path}  ({len(ppi_genes)} nodes)")

# Save base PPI edge list (before Pearson pruning — for reference)
ppi_path = DATA_PROC / 'ppi_edges.csv'
ppi.to_csv(ppi_path, index=False)
print(f"Base PPI edges -> {ppi_path}  ({len(ppi)} edges)")

# Save per-fold edge tensors
saved_folds = []
for fold_key, res in tqdm(fold_results.items(), desc='Saving fold tensors'):
    fold_dir = DATA_SPLITS / fold_key
    fold_dir.mkdir(parents=True, exist_ok=True)

    np.save(fold_dir / 'edge_index.npy',  res['edge_index'])    # (2, n_edges)
    np.save(fold_dir / 'edge_weight.npy', res['edge_weight'])   # (n_edges,)

    # Small metadata JSON per fold
    meta = {
        'fold_key'      : fold_key,
        'n_nodes'       : len(ppi_genes),
        'n_edges'       : int(res['n_edges_after']),
        'n_train_sids'  : int(res['n_train_sids']),
        'pearson_thresh': PEARSON_EDGE_THRESHOLD,
        'mean_abs_r'    : round(res['mean_abs_r'], 4),
    }
    with open(fold_dir / 'graph_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)
    saved_folds.append(fold_key)

print(f"\nSaved {len(saved_folds)} fold directories under {DATA_SPLITS}")
print("Example structure:")
example_dir = DATA_SPLITS / saved_folds[0]
for f in sorted(example_dir.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<25} {size:.0f} KB")

log.info(f"Per-fold tensors saved: {len(saved_folds)} folds")
print("\nCell 8 done. PPI phase complete.")
print("─" * 50)
print("Starting Reactome pathway mapping...")

Gene index map -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\processed\ppi_gene_index.csv  (4414 nodes)
Base PPI edges -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\processed\ppi_edges.csv  (37850 edges)



Saving fold tensors: 100%|██████████| 15/15 [00:00<00:00, 178.28it/s]
2026-07-06 23:06:24 | INFO     | phase5_graph | Per-fold tensors saved: 15 folds



Saved 15 fold directories under C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\splits
Example structure:
  edge_index.npy            115 KB
  edge_weight.npy           57 KB
  graph_meta.json           0 KB

Cell 8 done. PPI phase complete.
──────────────────────────────────────────────────
Starting Reactome pathway mapping...


---
## Cell 9 — Locate & Load Reactome Files

Expected files:
- `Ensembl2Reactome_All_Levels.txt` — Ensembl gene ID → Reactome pathway
- `ReactomePathways.txt` — Reactome pathway ID → pathway name
- `ReactomePathwaysRelation.txt` — pathway hierarchy (not used for filtering)

In [17]:
print(f"Scanning REACTOME dir: {REACTOME_DIR}")
print("Files found:")
for f in sorted(REACTOME_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1024**2:.1f} MB)")

REACTOME_E2P = find_file(REACTOME_DIR, [
    "Ensembl2Reactome_All_Levels.txt",
    "Ensembl2Reactome*.txt",
    "*Reactome_All_Levels*",
])
REACTOME_PATHS = find_file(REACTOME_DIR, [
    "ReactomePathways.txt",
    "*Pathways.txt",
])

print(f"\nEnsembl2Reactome : {REACTOME_E2P}")
print(f"ReactomePathways : {REACTOME_PATHS}")

# Load Ensembl → symbol map from Phase 1
ens_map_df = pd.read_csv(DATA_PROC / 'ensembl_to_symbol_map.csv', dtype=str)
ensembl_to_symbol = dict(zip(ens_map_df['ensembl_gene_id'], ens_map_df['symbol']))
print(f"\nPhase 1 Ensembl→symbol map : {len(ensembl_to_symbol):,} entries")

if not REACTOME_E2P:
    print("\nMISSING Ensembl2Reactome file. Check REACTOME_DIR.")
else:
    print("Reactome files found.")

Scanning REACTOME dir: C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\REACTOME
Files found:
  Ensembl2Reactome_All_Levels.txt  (480.7 MB)
  ReactomePathways.txt  (1.5 MB)
  ReactomePathwaysRelation.txt  (0.6 MB)

Ensembl2Reactome : C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\REACTOME\Ensembl2Reactome_All_Levels.txt
ReactomePathways : C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\REACTOME\ReactomePathways.txt

Phase 1 Ensembl→symbol map : 41,924 entries
Reactome files found.


---
## Cell 10 — Map Reactome Ensembl → HGNC → DEG Filter

Pipeline:
1. Load `Ensembl2Reactome_All_Levels.txt` (Homo sapiens rows only)
2. Strip Ensembl version suffixes
3. Map Ensembl → HGNC symbol using Phase 1 map
4. Filter to DEG members only
5. Build `pathway_id → set(gene_symbols)` dict
6. Filter pathways with < 5 or > 500 DEG members

Expected: **800–1,500 pathways** before Jaccard (higher than plan due to more DEGs).

In [18]:
print("Loading Ensembl2Reactome...")
t0 = time.time()

is_gz_r = str(REACTOME_E2P).endswith('.gz')
e2p = pd.read_csv(
    REACTOME_E2P,
    sep='\t',
    header=None,
    compression='gzip' if is_gz_r else None,
    names=['ensembl_id', 'reactome_id', 'url', 'pathway_name', 'evidence', 'species'],
    dtype=str
)
print(f"  All rows : {len(e2p):,}  ({time.time()-t0:.0f}s)")
print(f"  Species  : {e2p['species'].value_counts().head(4).to_string()}")

# Keep Homo sapiens only
e2p_hs = e2p[e2p['species'] == 'Homo sapiens'].copy()
print(f"  Homo sapiens rows : {len(e2p_hs):,}")

# Strip Ensembl version suffixes
e2p_hs['ensembl_id'] = e2p_hs['ensembl_id'].str.split('.').str[0]

# Map Ensembl → HGNC symbol
e2p_hs['gene_symbol'] = e2p_hs['ensembl_id'].map(ensembl_to_symbol)
mapped_hs = e2p_hs.dropna(subset=['gene_symbol'])
print(f"  After Ensembl→symbol map : {len(mapped_hs):,}")
print(f"  Mapping rate             : {len(mapped_hs)/len(e2p_hs)*100:.1f}%")

# Filter to DEG members only
deg_hs = mapped_hs[mapped_hs['gene_symbol'].isin(deg_set)].copy()
print(f"  After DEG filter         : {len(deg_hs):,}")

# Build pathway → gene set dict
pathway_genes_raw = (
    deg_hs.groupby('reactome_id')['gene_symbol']
    .agg(set)
    .to_dict()
)

# Also get pathway names
pathway_names = (
    deg_hs.drop_duplicates('reactome_id')
    .set_index('reactome_id')['pathway_name']
    .to_dict()
)

print(f"\nPathways before size filter : {len(pathway_genes_raw):,}")

# Filter by pathway size (DEG member count)
pathway_genes = {
    pid: genes
    for pid, genes in pathway_genes_raw.items()
    if PATHWAY_MIN_GENES <= len(genes) <= PATHWAY_MAX_GENES
}

size_dist = np.array([len(g) for g in pathway_genes.values()])
print(f"Pathways after size filter  : {len(pathway_genes):,}  (expected 800–1,500)")
print(f"DEG members per pathway:")
print(f"  Min    : {size_dist.min()}")
print(f"  Median : {np.median(size_dist):.0f}")
print(f"  Mean   : {size_dist.mean():.1f}")
print(f"  Max    : {size_dist.max()}")

log.info(f"Reactome: {len(pathway_genes)} pathways after size filter")
print("\nCell 10 done.")

Loading Ensembl2Reactome...
  All rows : 3,767,604  (9s)
  Species  : species
Homo sapiens        1126408
Mus musculus         371099
Sus scrofa           330556
Canis familiaris     326916
  Homo sapiens rows : 1,126,408
  After Ensembl→symbol map : 159,711
  Mapping rate             : 14.2%
  After DEG filter         : 44,703

Pathways before size filter : 2,372
Pathways after size filter  : 1,273  (expected 800–1,500)
DEG members per pathway:
  Min    : 5
  Median : 12
  Mean   : 26.2
  Max    : 469


2026-07-06 23:06:37 | INFO     | phase5_graph | Reactome: 1273 pathways after size filter



Cell 10 done.


---
## Cell 11 — Jaccard Redundancy Filtering

Iterative greedy algorithm (plan section 7.4):
- Sort pathways by DEG member count (largest first)
- Keep a pathway only if its Jaccard overlap with **all** already-kept pathways is < 0.7

Expected: **400–900 non-redundant pathways** after filtering.

**Runtime: ~1–5 min** depending on number of pathways.

In [19]:
def jaccard(s1, s2):
    union = s1 | s2
    return len(s1 & s2) / len(union) if union else 0.0

# Sort by size descending — larger pathways get priority
pathways_sorted = sorted(pathway_genes.items(), key=lambda x: -len(x[1]))

keep = []           # list of (pathway_id, gene_set) to keep
keep_sets = []      # just the gene sets for fast Jaccard comparison

print(f"Running Jaccard filter (threshold < {JACCARD_REDUNDANCY}) on {len(pathways_sorted):,} pathways...")
t0 = time.time()

for pid, genes in tqdm(pathways_sorted, desc='Jaccard dedup'):
    if all(jaccard(genes, kg) < JACCARD_REDUNDANCY for kg in keep_sets):
        keep.append((pid, genes))
        keep_sets.append(genes)

pathway_genes_filtered = dict(keep)
elapsed = time.time() - t0

n_before = len(pathway_genes)
n_after  = len(pathway_genes_filtered)
print(f"\nJaccard filter done in {elapsed:.0f}s")
print(f"Pathways before : {n_before:,}")
print(f"Pathways kept   : {n_after:,}  (expected 400–900)")
print(f"Pathways removed: {n_before - n_after:,}  ({(n_before-n_after)/n_before*100:.1f}% redundant)")

filt_sizes = np.array([len(g) for g in pathway_genes_filtered.values()])
print(f"\nKept pathway size (DEG members):")
print(f"  Min    : {filt_sizes.min()}")
print(f"  Median : {np.median(filt_sizes):.0f}")
print(f"  Max    : {filt_sizes.max()}")

# Show top 10 kept pathways
print(f"\nTop 10 largest kept pathways:")
for pid, genes in sorted(pathway_genes_filtered.items(), key=lambda x: -len(x[1]))[:10]:
    name = pathway_names.get(pid, pid)
    print(f"  {pid}  {len(genes):4d} DEGs  {name[:55]}")

log.info(f"Reactome Jaccard filter: {n_before} -> {n_after} pathways")
print("\nCell 11 done.")

Running Jaccard filter (threshold < 0.7) on 1,273 pathways...



Jaccard dedup: 100%|██████████| 1273/1273 [00:02<00:00, 592.85it/s]
2026-07-06 23:06:39 | INFO     | phase5_graph | Reactome Jaccard filter: 1273 -> 807 pathways



Jaccard filter done in 2s
Pathways before : 1,273
Pathways kept   : 807  (expected 400–900)
Pathways removed: 466  (36.6% redundant)

Kept pathway size (DEG members):
  Min    : 5
  Median : 12
  Max    : 469

Top 10 largest kept pathways:
  R-HSA-392499   469 DEGs  Metabolism of proteins
  R-HSA-168249   455 DEGs  Innate Immune System
  R-HSA-1280218   396 DEGs  Adaptive Immune System
  R-HSA-5663205   347 DEGs  Infectious disease
  R-HSA-109582   302 DEGs  Hemostasis
  R-HSA-372790   299 DEGs  Signaling by GPCR
  R-HSA-1280215   287 DEGs  Cytokine Signaling in Immune system
  R-HSA-382551   286 DEGs  Transport of small molecules
  R-HSA-74160   279 DEGs  Gene expression (Transcription)
  R-HSA-556833   215 DEGs  Metabolism of lipids

Cell 11 done.


---
## Cell 12 — Save Outputs + Audit YAML

In [20]:
# Run this in a new cell, then re-run Cell 12
degs_out_ppi = [g for g in deg_symbols if g not in set(ppi_genes)]
print(f"DEGs with no PPI edges : {len(degs_out_ppi):,}")

DEGs with no PPI edges : 5,583


In [21]:
def file_md5(path):
    h = hashlib.md5()
    with open(path,'rb') as f:
        while d := f.read(8192): h.update(d)
    return h.hexdigest()

# ── Save Reactome filtered ────────────────────────────────────────────────────
reactome_rows = [
    {'reactome_id': pid,
     'pathway_name': pathway_names.get(pid, ''),
     'n_deg_members': len(genes),
     'gene_symbols': '|'.join(sorted(genes))}
    for pid, genes in pathway_genes_filtered.items()
]
reactome_df  = pd.DataFrame(reactome_rows)
reactome_path = DATA_PROC / 'reactome_filtered.csv'
reactome_df.to_csv(reactome_path, index=False)
print(f"Reactome filtered -> {reactome_path}  ({len(reactome_df)} pathways)")

# ── Audit YAML ────────────────────────────────────────────────────────────────
fold_stats_audit = {
    fk: {
        'n_edges'     : int(v['n_edges_after']),
        'n_train_sids': int(v['n_train_sids']),
        'mean_abs_r'  : round(v['mean_abs_r'], 4),
    }
    for fk, v in fold_results.items()
}

audit = {
    'phase'     : 'Phase 5 — Graph Construction',
    'timestamp' : datetime.now().isoformat(),

    'ppi_subgraph': {
        'source'          : 'STRING v12 (human, 9606)',
        'score_threshold' : STRING_SCORE_THRESHOLD,
        'n_deg_nodes'     : len(ppi_genes),
        'n_base_edges'    : len(ppi),
        'degs_no_edges'   : len(degs_out_ppi),
        'edge_type'       : 'undirected, deduplicated',
    },

    'pearson_weights': {
        'method'          : 'Pearson |r| on training tumor VST per fold',
        'threshold'       : PEARSON_EDGE_THRESHOLD,
        'leakage_guard'   : 'Training tumor samples only (no val/test contamination)',
        'n_folds'         : len(fold_results),
        'mean_edges_kept' : int(np.mean([v['n_edges_after'] for v in fold_results.values()])),
        'fold_details'    : fold_stats_audit,
    },

    'reactome': {
        'source'           : 'Ensembl2Reactome_All_Levels.txt (Homo sapiens)',
        'size_filter'      : f'[{PATHWAY_MIN_GENES}, {PATHWAY_MAX_GENES}] DEG members',
        'jaccard_threshold': JACCARD_REDUNDANCY,
        'n_raw_pathways'   : len(pathway_genes_raw),
        'n_size_filtered'  : len(pathway_genes),
        'n_after_jaccard'  : len(pathway_genes_filtered),
    },

    'output_checksums': {
        'ppi_edges.csv'        : file_md5(ppi_path),
        'ppi_gene_index.csv'   : file_md5(gene_index_path),
        'reactome_filtered.csv': file_md5(reactome_path),
    },

    'next_phase' : 'Phase 6 — Feature Tensor Assembly (06_feature_tensor.ipynb)',
    'status'     : 'COMPLETE',
}

audit_path = AUDIT_DIR / 'phase5_graph_construction.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False)
print(f"Audit YAML -> {audit_path}")

print()
print("=" * 55)
print("PHASE 5 COMPLETE")
print("=" * 55)
print(f"  PPI nodes          : {len(ppi_genes):,}")
print(f"  PPI base edges     : {len(ppi):,}")
avg_kept = int(np.mean([v['n_edges_after'] for v in fold_results.values()]))
print(f"  Avg edges (|r|>={PEARSON_EDGE_THRESHOLD}) : {avg_kept:,}  (avg across 15 folds)")
print(f"  Reactome pathways  : {len(pathway_genes_filtered):,}  (after Jaccard dedup)")
print()
print("Next: 06_feature_tensor.ipynb — per-fold feature tensors (RNA z-score, CNV, degree)")

Reactome filtered -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\processed\reactome_filtered.csv  (807 pathways)
Audit YAML -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\audit\phase5_graph_construction.yaml

PHASE 5 COMPLETE
  PPI nodes          : 4,414
  PPI base edges     : 37,850
  Avg edges (|r|>=0.3) : 14,393  (avg across 15 folds)
  Reactome pathways  : 807  (after Jaccard dedup)

Next: 06_feature_tensor.ipynb — per-fold feature tensors (RNA z-score, CNV, degree)
